In [1]:
import os
import shutil
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download

# Raíz del proyecto: sube desde el directorio de trabajo hasta encontrar .git
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / ".git").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA = PROJECT_ROOT / "data" / "raw" / "datos_historicos_del_IPSA.csv"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
PROCESSED_PATH = PROCESSED_DATA / "ipsa_clean.parquet"


/Users/diegovidelasilva/DAVS_DataScience/Proyectos/proyecto_etl_cobre_dolar_chile/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(PROJECT_ROOT / ".env")

DATASET_URL = os.getenv("IPSA_DATASET_URL", "").strip()

if not DATASET_URL:
    raise ValueError("Falta IPSA_DATASET_URL en el archivo .env")

parts = [part for part in urlparse(DATASET_URL).path.split("/") if part]
if len(parts) > 5 and parts[0] == "datasets" and "resolve" in parts:
    resolve_index = parts.index("resolve")
    REPO_ID = "/".join(parts[1:resolve_index])
    URL_REVISION = "/".join(parts[resolve_index + 1 : -1]) or "main"
    FILENAME = parts[-1]
else:
    raise ValueError(f"URL de Hugging Face no reconocida: {DATASET_URL}")

REVISION = os.getenv("IPSA_DATASET_REVISION", "").strip() or URL_REVISION
FORCE_DOWNLOAD = False

# Asegurar que existan las carpetas antes de descargar
RAW_DATA.parent.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA.mkdir(parents=True, exist_ok=True)

if FORCE_DOWNLOAD or not RAW_DATA.exists() or RAW_DATA.stat().st_size == 0:
    cached_path = hf_hub_download(
        repo_id=REPO_ID,
        filename=FILENAME,
        repo_type="dataset",
        revision=REVISION,
    )
    temporary = RAW_DATA.with_name(f"{RAW_DATA.name}.part")
    shutil.copyfile(cached_path, temporary)
    temporary.replace(RAW_DATA)
    print(f"Dataset descargado desde Hugging Face: {REPO_ID} (revision: {REVISION})")
else:
    print("Dataset ya disponible en data/raw; se reutiliza el archivo local.")

print(f"Ruta: {RAW_DATA}")
print(f"Tamaño: {RAW_DATA.stat().st_size:,} bytes")


Dataset ya disponible en data/raw; se reutiliza el archivo local.
Ruta: /Users/diegovidelasilva/DAVS_DataScience/Proyectos/proyecto_etl_cobre_dolar_chile/data/raw/datos_historicos_del_IPSA.csv
Tamaño: 49,452 bytes


# Extraccion de dataset de fuentes

In [3]:
df = pd.read_csv(RAW_DATA, encoding="utf-8-sig")
# df_a (data frame analisis - data set de analisis) copia del dataset para mantener una copia original intacta (df)
df_a = df.copy()


In [4]:
df_a.head()

,Fecha,Último,Apertura,Máximo,Mínimo,Vol.,% var.
0,20.11.2026,"10.290,60","10.327,02","10.330,57","10.249,21",NaN,"-0,42%"
1,19.11.2026,"10.333,80","10.355,30","10.375,44","10.286,24",NaN,"-0,28%"
2,18.11.2026,"10.362,70","10.479,71","10.498,81","10.309,02",NaN,"-1,09%"
3,17.11.2026,"10.477,22","10.541,19","10.585,83","10.413,91",NaN,"-0,46%"
4,16.11.2026,"10.525,26","10.682,58","10.723,19","10.498,66",NaN,"-1,27%"


In [5]:
df_a.shape

(707, 7)

In [6]:
df_a.tail()

,Fecha,Último,Apertura,Máximo,Mínimo,Vol.,% var.
702,25.01.2024,"5.987,47","6.057,80","6.063,23","5.982,06",NaN,"-1,16%"
703,24.01.2024,"6.057,80","5.964,58","6.062,99","5.964,58",NaN,"1,56%"
704,23.01.2024,"5.964,58","5.921,04","5.985,63","5.921,04",NaN,"0,74%"
705,22.01.2024,"5.921,04","5.844,56","5.926,09","5.832,36",NaN,"1,31%"
706,19.01.2024,"5.844,56","5.867,72","5.894,31","5.822,96",NaN,"-0,39%"


# Analisis de DataSet

In [7]:
df_a.columns

Index(['Fecha', 'Último', 'Apertura', 'Máximo', 'Mínimo', 'Vol.', '% var.'],
      dtype='str')

In [8]:
df_a.info()

<class 'pandas.DataFrame'>
RangeIndex: 707 entries, 0 to 706
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Fecha     707 non-null    str    
 1   Último    707 non-null    str    
 2   Apertura  707 non-null    str    
 3   Máximo    707 non-null    str    
 4   Mínimo    707 non-null    str    
 5   Vol.      0 non-null      float64
 6   % var.    707 non-null    str    
dtypes: float64(1), str(6)
memory usage: 72.5 KB


In [9]:
df_a.describe(include='all')

,Fecha,Último,Apertura,Máximo,Mínimo,Vol.,% var.
count,707,707,707,707,707,0.0,707
unique,707,706,706,698,703,NaN,311
top,20.11.2026,"6.374,35","6.374,35","11.025,21","10.691,27",NaN,"-0,09%"
freq,1,2,2,2,2,NaN,10
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
df_a.isna().sum()

Fecha         0
Último        0
Apertura      0
Máximo        0
Mínimo        0
Vol.        707
% var.        0
dtype: int64

In [11]:
df_a.duplicated().sum()

np.int64(0)

In [12]:
df_a['Fecha'].value_counts(dropna=False)

Fecha
20.11.2026    1
19.11.2026    1
18.11.2026    1
17.11.2026    1
16.11.2026    1
             ..
25.01.2024    1
24.01.2024    1
23.01.2024    1
22.01.2024    1
19.01.2024    1
Name: count, Length: 707, dtype: int64

# Limpieza de Dataset

In [13]:
# nueva copia del dataset, de df_a de analisis para a df_w de work

# reemplazar el nombre de columna "% Var" por el nombre completo para mayor claridad "%_variacion"
# y tambien normalizar los otros nombre de columnas quitando mayusculas y acentos
df_w = df_a.rename(columns=
            {'% var.' : 'porcentaje_variacion',
             'Fecha' : 'fecha',
             'Último' : 'ultimo',
             'Apertura' : 'apertura',
             'Máximo' : 'maximo',
             'Mínimo' : 'minimo'

             })
df_w.columns


Index(['fecha', 'ultimo', 'apertura', 'maximo', 'minimo', 'Vol.',
       'porcentaje_variacion'],
      dtype='str')

In [14]:
# La columna 'Vol' solo contiene NaN por lo que sera eliminada
df_w = df_w.drop(columns='Vol.')
df_w.columns


Index(['fecha', 'ultimo', 'apertura', 'maximo', 'minimo',
       'porcentaje_variacion'],
      dtype='str')

In [15]:
df_w.dtypes

fecha                   str
ultimo                  str
apertura                str
maximo                  str
minimo                  str
porcentaje_variacion    str
dtype: object

In [16]:
# Conversion de tipos de datos

df_w['fecha'] = pd.to_datetime(df_w['fecha'], format='%d.%m.%Y', errors='coerce')

cols_numericas = ['ultimo', 'apertura', 'maximo', 'minimo']
for col in cols_numericas:
    df_w[col] = pd.to_numeric(
        df_w[col].astype(str).str.strip().str.replace('.', '', regex=False).str.replace(',', '.', regex=False),
        errors='coerce'
    )

df_w['porcentaje_variacion'] = pd.to_numeric(
    df_w['porcentaje_variacion']
    .astype(str)
    .str.strip()
    .str.replace('%', '', regex=False)
    .str.replace(',', '.', regex=False),
    errors='coerce'
)


In [17]:
df_w.dtypes

fecha                   datetime64[us]
ultimo                         float64
apertura                       float64
maximo                         float64
minimo                         float64
porcentaje_variacion           float64
dtype: object

In [18]:
df_w.head()

,fecha,ultimo,apertura,maximo,minimo,porcentaje_variacion
0,2026-11-20,10290.60,10327.02,10330.57,10249.21,-0.42
1,2026-11-19,10333.80,10355.30,10375.44,10286.24,-0.28
2,2026-11-18,10362.70,10479.71,10498.81,10309.02,-1.09
3,2026-11-17,10477.22,10541.19,10585.83,10413.91,-0.46
4,2026-11-16,10525.26,10682.58,10723.19,10498.66,-1.27


In [19]:
df_w.isna().sum()

fecha                   0
ultimo                  0
apertura                0
maximo                  0
minimo                  0
porcentaje_variacion    0
dtype: int64

In [20]:
df_w['dia'] = df_w['fecha'].dt.day
df_w['mes'] = df_w['fecha'].dt.month
df_w['anio'] = df_w['fecha'].dt.year

TRADUCCION = {
    'Monday': 'lunes',
    'Tuesday': 'martes',
    'Wednesday': 'miercoles',
    'Thursday': 'jueves',
    'Friday': 'viernes',
    'Saturday': 'sabado',
    'Sunday': 'domingo',
}

df_w['dia_semana'] = df_w['fecha'].dt.day_name().map(TRADUCCION)


In [21]:
df_w.head()

,fecha,ultimo,apertura,maximo,minimo,porcentaje_variacion,dia,mes,anio,dia_semana
0,2026-11-20,10290.60,10327.02,10330.57,10249.21,-0.42,20,11,2026,viernes
1,2026-11-19,10333.80,10355.30,10375.44,10286.24,-0.28,19,11,2026,jueves
2,2026-11-18,10362.70,10479.71,10498.81,10309.02,-1.09,18,11,2026,miercoles
3,2026-11-17,10477.22,10541.19,10585.83,10413.91,-0.46,17,11,2026,martes
4,2026-11-16,10525.26,10682.58,10723.19,10498.66,-1.27,16,11,2026,lunes


In [22]:
df_w = df_w.sort_values('fecha').reset_index(drop=True)

### Resumen:

In [23]:
print('Resumen del Dataset Work luego de transformaciones y limpieza')
print('=' * 61)
print('DF_Analysis: ', df_a.shape, "-> DF_Work: ", df_w.shape)
print('=' * 61)
print('Columnas antes: ', list(df_a.columns))
print('Columnas después:', list(df_w.columns))
print('=' * 61)
print('DF_Work tiene: ', int(df_w.isna().sum().sum()), ' Valores Nulos')
print('DF_Work tiene: ', int(df_w.duplicated().sum()), ' Valores Duplicados')
print('DF_Work tiene: ', int(df_w['fecha'].duplicated().sum()), ' Fechas Duplicadas')
print('=' * 61)
print('Rango de fechas: ', df_w['fecha'].min().date(), '->', df_w['fecha'].max().date())
print('Días únicos: ', df_w['fecha'].nunique())
print('=' * 61)
print('Tipos de datos finales:')
print(df_w.dtypes.to_string())
print('=' * 61)
print('Rangos de variables numéricas:')
columnas_excluidas = {'anio', 'mes', 'dia'}
columnas_numericas = [
    columna for columna in df_w.select_dtypes(include='number').columns
    if columna not in columnas_excluidas
]
for columna in columnas_numericas:
    print(f'  {columna}: min={df_w[columna].min():,.4f} | max={df_w[columna].max():,.4f}')
print('=' * 61)


Resumen del Dataset Work luego de transformaciones y limpieza
DF_Analysis:  (707, 7) -> DF_Work:  (707, 10)
Columnas antes:  ['Fecha', 'Último', 'Apertura', 'Máximo', 'Mínimo', 'Vol.', '% var.']
Columnas después: ['fecha', 'ultimo', 'apertura', 'maximo', 'minimo', 'porcentaje_variacion', 'dia', 'mes', 'anio', 'dia_semana']
DF_Work tiene:  0  Valores Nulos
DF_Work tiene:  0  Valores Duplicados
DF_Work tiene:  0  Fechas Duplicadas
Rango de fechas:  2024-01-19 -> 2026-11-20
Días únicos:  707
Tipos de datos finales:
fecha                   datetime64[us]
ultimo                         float64
apertura                       float64
maximo                         float64
minimo                         float64
porcentaje_variacion           float64
dia                              int32
mes                              int32
anio                             int32
dia_semana                         str
Rangos de variables numéricas:
  ultimo: min=5,844.5600 | max=11,627.5800
  apertura: min=5,

In [24]:
df_w = df_w[['fecha', 'anio', 'mes', 'dia', 'dia_semana', 'ultimo', 'apertura', 'maximo', 'minimo', 'porcentaje_variacion']]
df_w.head()

,fecha,anio,mes,dia,dia_semana,ultimo,apertura,maximo,minimo,porcentaje_variacion
0,2024-01-19,2024,1,19,viernes,5844.56,5867.72,5894.31,5822.96,-0.39
1,2024-01-22,2024,1,22,lunes,5921.04,5844.56,5926.09,5832.36,1.31
2,2024-01-23,2024,1,23,martes,5964.58,5921.04,5985.63,5921.04,0.74
3,2024-01-24,2024,1,24,miercoles,6057.80,5964.58,6062.99,5964.58,1.56
4,2024-01-25,2024,1,25,jueves,5987.47,6057.80,6063.23,5982.06,-1.16


In [25]:
# Guardar el dataset transformado en formato Parquet
PROCESSED_DATA.mkdir(parents=True, exist_ok=True)
temporary = PROCESSED_PATH.with_name(f"{PROCESSED_PATH.name}.part")
df_w.to_parquet(temporary, index=False, engine="pyarrow")
temporary.replace(PROCESSED_PATH)

print(f"Dataset procesado: {PROCESSED_PATH}")


Dataset procesado: /Users/diegovidelasilva/DAVS_DataScience/Proyectos/proyecto_etl_cobre_dolar_chile/data/processed/ipsa_clean.parquet
